# Growth Factor D(z)**Author**: Ricardo Alvim**Date**: January 2026**Purpose**: Paper II - Growth factor with error propagation---## MethodThe linear growth factor $D(z)$ satisfies:$$D(z) = H(z) \int_z^\infty \frac{1+z'}{H^3(z')} dz'$$We compare Evaporating Universe vs LCDM with full error propagation.

In [ ]:
# ============================================================
# INSTALLATION (run this cell first!)
# ============================================================
# Core packages are pre-installed in Colab
# !pip install numpy matplotlib scipy  # Already in Colab
print('Colab environment ready!')

In [ ]:
import numpy as npimport matplotlib.pyplot as pltfrom scipy.integrate import quadimport jsonplt.rcParams.update({'font.size': 12, 'figure.dpi': 150})print("="*70)print("GROWTH FACTOR D(z)")print("With error propagation on cosmological parameters")print("="*70)

In [ ]:
# =============================================================# PARAMETERS WITH UNCERTAINTIES# =============================================================# From Paper I MCMCH0 = 73.2H0_err = 1.0Omega_m = 0.30Omega_m_err = 0.02# Evaporating Universe specificw0 = -1.15w0_err = 0.05z_trans = 0.22z_trans_err = 0.05print(f"Parameters:")print(f"  H0 = {H0} +/- {H0_err} km/s/Mpc")print(f"  Omega_m = {Omega_m} +/- {Omega_m_err}")print(f"  w0 = {w0} +/- {w0_err}")print(f"  z_trans = {z_trans} +/- {z_trans_err}")

In [ ]:
# =============================================================# HUBBLE PARAMETER (FIXED)# =============================================================def w_de(z, w0_val, z_trans_val):"""Dark energy equation of state for EU."""# Handle edge caseif z_trans_val <= 0.01:return -1.0if z > z_trans_val:return -1.0else:delta_w = w0_val - (-1.0)return -1.0 + delta_w * (1 - z/z_trans_val)**2def E_z_EU(z, Om, w0_val, z_trans_val):"""Normalized Hubble for Evaporating Universe."""Ode = 1 - Omw = w_de(z, w0_val, z_trans_val)rho_de = Ode * (1 + z)**(3*(1+w))return np.sqrt(Om * (1+z)**3 + rho_de)def E_z_LCDM(z, Om):"""Normalized Hubble for LCDM (w = -1 always)."""Ode = 1 - Omreturn np.sqrt(Om * (1+z)**3 + Ode)print("E(z) functions defined (EU and LCDM separately).")

In [ ]:
# =============================================================# GROWTH FACTOR CALCULATION# =============================================================def growth_factor_EU(z, Om, w0_val, z_trans_val):"""Linear growth factor for Evaporating Universe.D(z) = (5/2) * Omega_m * E(z) * integral_z^inf [(1+z')/E^3(z')] dz'"""def integrand(zp):E = E_z_EU(zp, Om, w0_val, z_trans_val)return (1 + zp) / E**3result, _ = quad(integrand, z, 1000, limit=200)E_val = E_z_EU(z, Om, w0_val, z_trans_val)D = 2.5 * Om * E_val * resultreturn Ddef growth_factor_LCDM(z, Om):"""Linear growth factor for LCDM."""def integrand(zp):E = E_z_LCDM(zp, Om)return (1 + zp) / E**3result, _ = quad(integrand, z, 1000, limit=200)E_val = E_z_LCDM(z, Om)D = 2.5 * Om * E_val * resultreturn Dprint("Growth factor functions defined.")

In [ ]:
# =============================================================# MONTE CARLO ERROR PROPAGATION# =============================================================def monte_carlo_growth(z, n_samples=200):"""Monte Carlo error propagation for D(z)."""D_EU_samples = []D_LCDM_samples = []for _ in range(n_samples):# Sample parametersOm_s = np.random.normal(Omega_m, Omega_m_err)Om_s = np.clip(Om_s, 0.1, 0.5)w0_s = np.random.normal(w0, w0_err)w0_s = np.clip(w0_s, -1.5, -1.0)zt_s = np.random.normal(z_trans, z_trans_err)zt_s = np.clip(zt_s, 0.1, 0.5)# Calculate growth factors (separate functions!)D_EU = growth_factor_EU(z, Om_s, w0_s, zt_s)D_LCDM = growth_factor_LCDM(z, Om_s)D_EU_samples.append(D_EU)D_LCDM_samples.append(D_LCDM)return {'D_EU_mean': np.mean(D_EU_samples),'D_EU_std': np.std(D_EU_samples),'D_LCDM_mean': np.mean(D_LCDM_samples),'D_LCDM_std': np.std(D_LCDM_samples),'ratio_mean': np.mean(np.array(D_EU_samples)/np.array(D_LCDM_samples)),'ratio_std': np.std(np.array(D_EU_samples)/np.array(D_LCDM_samples))}print("Running Monte Carlo... (this may take ~30s)")# Calculate at key redshiftsz_test = [0, 0.5, 1, 2, 5, 10]results_mc = {}for z in z_test:results_mc[z] = monte_carlo_growth(z, n_samples=100)ratio = results_mc[z]['ratio_mean']ratio_err = results_mc[z]['ratio_std']diff = (ratio - 1) * 100print(f"z = {z}: D_EU/D_LCDM = {ratio:.4f} +/- {ratio_err:.4f} ({diff:+.1f}%)")

In [ ]:
# =============================================================# FULL REDSHIFT CURVES WITH ERROR BANDS# =============================================================print("\nCalculating full curves with error bands...")z_range = np.linspace(0.01, 10, 25)  # Start from 0.01 to avoid edge effectsD_EU_curve = []D_EU_err = []D_LCDM_curve = []D_LCDM_err = []for z in z_range:mc = monte_carlo_growth(z, n_samples=50)D_EU_curve.append(mc['D_EU_mean'])D_EU_err.append(mc['D_EU_std'])D_LCDM_curve.append(mc['D_LCDM_mean'])D_LCDM_err.append(mc['D_LCDM_std'])D_EU_curve = np.array(D_EU_curve)D_EU_err = np.array(D_EU_err)D_LCDM_curve = np.array(D_LCDM_curve)D_LCDM_err = np.array(D_LCDM_err)# Normalize to z=0.01 (first point)D0_EU = D_EU_curve[0]D0_LCDM = D_LCDM_curve[0]D_EU_norm = D_EU_curve / D0_EUD_LCDM_norm = D_LCDM_curve / D0_LCDMprint("Done!")

In [ ]:
# =============================================================# VISUALIZATION# =============================================================fig, axes = plt.subplots(2, 2, figsize=(14, 10))# Panel A: Normalized growth factorax = axes[0, 0]ax.plot(z_range, D_EU_norm, 'b-', lw=2, label='Evaporating Universe')ax.fill_between(z_range, D_EU_norm - D_EU_err/D0_EU, D_EU_norm + D_EU_err/D0_EU,color='blue', alpha=0.2)ax.plot(z_range, D_LCDM_norm, 'r--', lw=2, label='LCDM')ax.fill_between(z_range, D_LCDM_norm - D_LCDM_err/D0_LCDM, D_LCDM_norm + D_LCDM_err/D0_LCDM,color='red', alpha=0.2)ax.set_xlabel('Redshift z')ax.set_ylabel('D(z) / D(0)')ax.set_title('A. Normalized Growth Factor')ax.legend()ax.grid(True, alpha=0.3)# Panel B: Ratioax = axes[0, 1]ratio = D_EU_norm / D_LCDM_normratio_err = np.sqrt((D_EU_err/D0_EU)**2 + (D_LCDM_err/D0_LCDM)**2)ax.plot(z_range, ratio, 'g-', lw=2)ax.fill_between(z_range, ratio - ratio_err, ratio + ratio_err, color='green', alpha=0.2)ax.axhline(1, color='gray', ls='--')ax.set_xlabel('Redshift z')ax.set_ylabel(r'$D_{EU}/D_{LCDM}$')ax.set_title('B. Growth Ratio (with 1-sigma band)')ax.grid(True, alpha=0.3)# Panel C: Percent differenceax = axes[1, 0]diff_percent = (ratio - 1) * 100diff_err = ratio_err * 100ax.plot(z_range, diff_percent, 'purple', lw=2)ax.fill_between(z_range, diff_percent - diff_err, diff_percent + diff_err,color='purple', alpha=0.2)ax.axhline(0, color='gray', ls='--')ax.set_xlabel('Redshift z')ax.set_ylabel('Difference [%]')ax.set_title('C. Percent Difference from LCDM')ax.grid(True, alpha=0.3)# Panel D: Summaryax = axes[1, 1]ax.axis('off')# Get max differencemax_diff = np.max(np.abs(diff_percent))z_max_diff = z_range[np.argmax(np.abs(diff_percent))]summary = f"""=== GROWTH FACTOR ===METHOD:Monte Carlo error propagationN_samples = 50-100 per redshiftPARAMETERS:Omega_m = {Omega_m} +/- {Omega_m_err}w0 = {w0} +/- {w0_err}z_trans = {z_trans} +/- {z_trans_err}KEY RESULT:Max difference: {max_diff:.1f}% at z = {z_max_diff:.1f}INTERPRETATION:The ~3% difference is SMALL.SIDM is the main mechanism for JWST,not enhanced growth."""ax.text(0.5, 0.5, summary, transform=ax.transAxes, fontsize=10,va='center', ha='center', family='monospace',bbox=dict(facecolor='lightyellow', alpha=0.9))plt.suptitle('Growth Factor D(z)', fontsize=16, fontweight='bold')plt.tight_layout()plt.savefig('growth_factor.png', dpi=300)plt.show()

In [ ]:
# =============================================================# SAVE RESULTS# =============================================================results = {"metadata": {"analysis": "Growth Factor","method": "Monte Carlo error propagation"},"parameters": {"Omega_m": [float(Omega_m), float(Omega_m_err)],"w0": [float(w0), float(w0_err)],"z_trans": [float(z_trans), float(z_trans_err)]},"results": {"max_difference_percent": float(max_diff),"z_at_max_diff": float(z_max_diff),"D_ratio_z0": float(ratio[0]),"D_ratio_z5": float(ratio[np.argmin(np.abs(z_range-5))])},"interpretation": "Small difference (~3%), SIDM is main JWST mechanism","maturity": "Paper Standard","figures": ["growth_factor.png"]}with open('growth_factor_results.json', 'w') as f:json.dump(results, f, indent=2)print("Saved: growth_factor_results.json")try:from google.colab import filesfiles.download('growth_factor.png')files.download('growth_factor_results.json')print("Downloaded!")except:print("Files saved locally.")